In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

df = pd.read_csv(r"D:\manager-bounce\data\matched_pairs.csv")
print(df.shape)
df.head()

(108, 13)


,treated_club_id,treated_season,treated_match_no,treated_ppg_prior,treated_ppg_next,treated_position,control_club_id,control_season,control_match_no,control_ppg_prior,control_ppg_next,control_position,match_dist
0,533,2012,18,0.5,1.000000,16,79,2013,28,0.5,1.333333,16,0.0
1,65,2012,23,0.5,0.800000,18,23,2013,9,0.5,0.800000,18,0.0
2,533,2012,28,0.8,1.333333,17,167,2012,21,0.8,1.500000,17,0.0
3,79,2013,4,0.0,1.300000,14,44,2022,2,0.0,1.000000,14,0.0
4,41,2013,6,0.8,1.200000,16,60,2013,12,0.8,1.000000,16,0.0


In [2]:
!pip install statsmodels scipy

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------------- 0.5/11.3 MB 241.0 kB/s eta 0:00:45
   - -------------------------------

In [3]:
df['treated_change'] = df['treated_ppg_next'] - df['treated_ppg_prior']
df['control_change'] = df['control_ppg_next'] - df['control_ppg_prior']

print("Mean treated change:", df['treated_change'].mean())
print("Mean control change:", df['control_change'].mean())
print("Gap:", df['treated_change'].mean() - df['control_change'].mean())

Mean treated change: 0.6433568489124044
Mean control change: 0.5263043797766019
Gap: 0.11705246913580247


In [4]:
t_stat, p_val = stats.ttest_rel(df['treated_change'], df['control_change'])
print(f"Paired t-test: t = {t_stat:.4f}, p = {p_val:.4f}")

Paired t-test: t = 1.8919, p = 0.0612


In [5]:
long = pd.DataFrame({
    'change': pd.concat([df['treated_change'], df['control_change']]),
    'treated': [1]*len(df) + [0]*len(df),
    'pair_id': list(df.index) * 2
})

model = smf.ols('change ~ treated', data=long).fit(
    cov_type='cluster', cov_kwds={'groups': long['pair_id']}
)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 change   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     3.563
Date:                Fri, 25 Sep 2026   Prob (F-statistic):             0.0618
Time:                        00:15:49   Log-Likelihood:                -173.62
No. Observations:                 216   AIC:                             351.2
Df Residuals:                     214   BIC:                             358.0
Df Model:                           1                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5263      0.053      9.871      0.0

In [6]:
df5 = pd.read_csv(r"D:\manager-bounce\data\matched_pairs_5.csv")
df5['treated_change'] = df5['treated_ppg_next'] - df5['treated_ppg_prior']
df5['control_change'] = df5['control_ppg_next'] - df5['control_ppg_prior']

print("Mean treated change:", df5['treated_change'].mean())
print("Mean control change:", df5['control_change'].mean())
print("Gap:", df5['treated_change'].mean() - df5['control_change'].mean())

t_stat, p_val = stats.ttest_rel(df5['treated_change'], df5['control_change'])
print(f"Paired t-test: t = {t_stat:.4f}, p = {p_val:.4f}")

Mean treated change: 0.7729691876750702
Mean control change: 0.5421652421652423
Gap: 0.23080394550982786
Paired t-test: t = nan, p = nan


In [7]:
print(df5['treated_change'].isna().sum())
print(df5['control_change'].isna().sum())
print(df5.shape)

0
2
(119, 15)


In [8]:
before = len(df5)
df5_clean = df5.dropna(subset=['treated_change', 'control_change'])
after = len(df5_clean)
print(f"Dropped {before - after} pairs with missing values (likely season-boundary matches)")

t_stat, p_val = stats.ttest_rel(df5_clean['treated_change'], df5_clean['control_change'])
print(f"Paired t-test (n={after}): t = {t_stat:.4f}, p = {p_val:.4f}")

print("Mean treated change:", df5_clean['treated_change'].mean())
print("Mean control change:", df5_clean['control_change'].mean())
print("Gap:", df5_clean['treated_change'].mean() - df5_clean['control_change'].mean())

Dropped 2 pairs with missing values (likely season-boundary matches)
Paired t-test (n=117): t = 2.8575, p = 0.0051
Mean treated change: 0.7673789173789175
Mean control change: 0.5421652421652422
Gap: 0.2252136752136753
